## Import libraries

In [76]:
import os
import re
import gc
import json
import math
import time
import logging
from dataclasses import dataclass, asdict
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## Enviroment

In [77]:
# ---------------------------
# 0) Environment knobs
# ---------------------------
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "0")
os.environ.setdefault("TORCH_CUDNN_V8_API_ENABLED", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("byt-robust")

def print_environment_info():
    logger.info(f"torch={torch.__version__}")
    logger.info(f"cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        logger.info(f"gpu={torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        logger.info(f"gpu_mem_total_gb={props.total_memory/1024**3:.2f}")
    # BetterTransformer availability (optional)
    try:
        from optimum.bettertransformer import BetterTransformer  # noqa: F401
        logger.info("optimum.bettertransformer is available (will try to use if enabled).")
    except Exception:
        logger.info("optimum.bettertransformer not available (will skip).")

print_environment_info()

2026-03-13 15:16:21,908 | INFO | torch=2.8.0+cu126
2026-03-13 15:16:21,909 | INFO | cuda_available=True
2026-03-13 15:16:21,910 | INFO | gpu=Tesla T4
2026-03-13 15:16:21,911 | INFO | gpu_mem_total_gb=14.56
2026-03-13 15:16:21,914 | INFO | optimum.bettertransformer not available (will skip).


## Configs

In [78]:
@dataclass
class CFG:
    # Paths
    test_path = "/kaggle/input/deep-past-initiative-machine-translation/test.csv" # test path
    train_path = "/kaggle/input/deep-past-initiative-machine-translation/train.csv"
    model_path = "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x" # path of ByT5 model
    output_dir = "/kaggle/working/submission.csv" # submit path

    # Tokenizations
    max_length = 512 # max input token length
    batch_size = 8
    num_workers = 2
    pin_memory = True # speed up memory transfer 

    # Beam Search settings
    num_beams = 8 # check 8 translation candidates
    max_new_tokens = 512 # maximum output
    length_penalty = 1.3 # beam search scoring prefers longer sequences
    early_stopping = True
    no_repeat_ngram_size = 0 # prevent repeating
    repetition_penalty = None # ++penalty if repeating words

    # Perf. optimizations
    use_mixed_precision = True # enable mixed precision --> lower computations
    use_better_transformer = True # idk what is that
    use_bucket_batching = True # split instances with similar length to the batch --> less padding
    use_adaptive_beams = True # adjust beam size (longer --> higher num_beam)
    use_vectorized_postproc = True # idk what is that x2

    # Operations conf.
    checkpoint_freq = 2000 # save partial submission per 2000 sample (kernel crash -> resume process)
    empty_cache_every = 10 # torch.cuda.empty_cache() every 10 batches
    validate_samples = 6 # number of instances logged for sanity check

    # Postprocess
    postprocess_mode = "safe_then_conditional_aggressive"
    '''
    safe_only: basic cleanup
    aggressive_only: heavy modifications
    safe_then_conditional_aggressive: safe first, aggressive if output looks bad
    '''
    aggressive_trigger_badness = 2.5 # if badness >= threshold -> run aggressive refinement 
    min_words_fallback = 3

    # Device selection
    device = "cuda" if torch.cuda.is_available() else "cpu" 

## Preprocessor

Includes

- small gap, big gap convertor
- normalize spaces

In [79]:
# preprocess the translit. texts
class Preprocessor:
    def __init__(self):
        # gaps def.: Some of ancient tablets has missing characters.
        # big gaps: ... … …… etc 
        self.big_gap = re.compile(r"\.\.\.+|…+") # 3+ "." or 1+ "…" -> <big_gap>
        # small gaps: occurrences of 'x' / 'xx' in isolation-ish patterns
        # This is intentionally conservative; you can expand if needed.
        self.small_gap = re.compile(r"(?:(?<=\s)|^)(x{1,3})(?=(\s|$))", flags=re.IGNORECASE)
        # detect small unknown word marked as letter x
        # include standalone x token, 1->3 "x", ensure a space and end after

    def preprocess_batch(self, texts: List[str]) -> List[str]:
        # preprocessing a batch <texts> from raw List -> preprocessed List (cleaned)
        s = pd.Series(texts, dtype="object").fillna("") # convert text -> pd.Series --> faster
        s = s.str.replace(self.big_gap, " <big_gap> ", regex=True) # fill big_gap in def. to <big_gap>
        s = s.str.replace(self.small_gap, " <gap> ", regex=True) # fill small gap in def. to <gap>
        s = s.str.replace(r"\s+", " ", regex=True).str.strip() # Space normalization
        s = s.tolist() # return pd.Series -> List
        return s

## Postprocessor

### Safe Postprocessor

Only contain cleaning format, not editing contents (rewrite sentences)

In [80]:
class SafePostprocessor:
    """
    Conservative cleanup.
    - Preserve common punctuation and formatting.
    - Protect <gap>/<big_gap>.
    - Normalize a few known characters.
    """
    SUBSCRIPT_TO_NORMAL = str.maketrans({
        "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4","₅":"5","₆":"6","₇":"7","₈":"8","₉":"9"
    }) # change to CDLI

    def __init__(self, use_unicode_fractions=False, strip_dash_old=False):
        # use_unicode_fractions: convert 1/4, 1/2, 3/4 to ¼, ½, ¾.
        # strip_dash_old: remove dashes ("- text -" → "text")
        self.use_unicode_fractions = use_unicode_fractions
        self.strip_dash_old = strip_dash_old
        
        self.forbidden_chars = re.compile(r"[⌈⌋⌊⌉]") # unsafe characters
        self.multi_space = re.compile(r"\s+") # Normalize spaces
        self.space_before_punct = re.compile(r"\s+([,.;:!?])") # remove spaces before punctuations
        self.multi_punct = re.compile(r"([.?!])\1{2,}") # replace multiple punct.
        self.dashes = re.compile(r"[–—]") # Normalize dashes

        # <gap> and <big_gap> placeholders (restore latter)
        self.prot_gap = "\uFFF0"
        self.prot_big = "\uFFF1"

        # Gap Normalization
        # model may genererate gaps in different formats
        self.bracket_x = re.compile(r"\[\s*x\s*\]|\(\s*x\s*\)", re.IGNORECASE) # [x], (x) -> <gap>
        self.bare_x = re.compile(r"(?:(?<=\s)|^)x(?=(\s|$))", re.IGNORECASE)
        # text x text -> text <gap> text
        self.ellipsis = re.compile(r"(\.\.\.+|…+)") # ..., … -> <big_gap>

        # Optional fractions
        self.frac_map = {
            "0.5": "½", "0.25": "¼", "0.75": "¾",
            "1/2": "½", "1/4": "¼", "3/4": "¾",
        }
        # apply if use_unicode_fractions enabled
        self.frac_re = re.compile(r"\b(0\.5|0\.25|0\.75|1/2|1/4|3/4)\b")

    def postprocess_one(self, text: str) -> str:
        if text is None:
            text = "" # avoid error
        t = str(text)
        t = t.replace("ḫ","h") # simplify "h"
        t = t.translate(self.SUBSCRIPT_TO_NORMAL) # subscript num -> normal
        t = self.dashes.sub("-", t) # apply normalize dashes

        # Apply normalize gaps
        t = self.bracket_x.sub("<gap>", t)
        t = self.bare_x.sub("<gap>", t)
        t = self.ellipsis.sub("<big_gap>", t)

        # Protect tokens -> later cleaning not modify gaps
        t = t.replace("<gap>", self.prot_gap)
        t = t.replace("<big_gap>", self.prot_big)

        # Remove unsafe char.
        t = self.forbidden_chars.sub("", t)

        # Restore gaps after remove unsafe chars
        t = t.replace(self.prot_gap, "<gap>")
        t = t.replace(self.prot_big, "<big_gap>")

        # Optional dash striping
        if self.strip_dash_old:
            t.strip(" -") # - text - --> text

        # Optional unicode fractions
        if self.use_unicode_fractions:
            t = self.frac_re.sub(lambda m: self.frac_map.get(m.group(1), m.group(1)), t) # apply

        # clean whitespace and punctuation
        t = self.space_before_punct.sub(r"\1", t)
        t = self.multi_punct.sub(r"\1", t)
        t = self.multi_space.sub(" ", t).strip()

    def postprocess_batch(self, texts):
        return[self.postprocess_one(text) for text in texts]
            

### Aggressive Postprocessor

Use for bad translation (badness >= threshold), include:

- Remove grammatical notes
- Remove repeated words
- Remove repeated phrases (n-grams)
- Consolidate gaps

In [81]:
class AgressivePostprocessor:
    def __init__(self, ngram_dedupe_max_n=4):
        self.ngram_dedupe_max_n = ngram_dedupe_max_n # dedupe repeated 4-word phrases

        # Some regex rules to detect issues
        self.multi_space = re.compile(r"\s+") # Multiple space issue
        self.space_before_punct = re.compile(r"\s+([,.;:!?])") # Space before punct. issue
        self.multi_punct = re.compile(r"([!?.,])\1{2,}") # Multiple punct. issue
        self.remove_notes = re.compile(
            r"\((?:plur\.?|sing\.?|fem\.?|masc\.?|uncertain|\?|\!|damaged|broken)\)",
            flags=re.IGNORECASE
        ) # Has annotations issue (Kaggle output don't have this)
        self.remove_weird = re.compile(r"[<>⌈⌋⌊⌉⌊⌋\+ʾ/;]") # remove "< > ⌈ ⌋ ⌊ + ʾ / ;" (aggressive)

        # Gap merger
        self.gap_runs = re.compile(r"(?:<gap>\s*){2,}") # <gap> <gap>
        self.biggap_runs = re.compile(r"(?:<big_gap>\s*){2,}") # <big_gap> <big_gap>

        # Remove repeated words (word word word word2 --> word word2)
        self.repeat_word = re.compile(r"\b(\w+)(\s+\1){1,}\b", flags=re.IGNORECASE)

    # Remove repeated phrases
    def _dedupe_ngrams(self, text: str) -> str:
        # simple greedy n-gram dedupe for 2..N
        tokens = text.split()
        if len(tokens) < 12:
            return text
        for n in range(1, ngram_dedupe_max_n + 1):
            i = 0
            out = []
            while i < len(tokens):
                if tokens[i:i+n] == tokens[i+n:i+2*n] and i + 2*n < len(tokens):
                    out.extend(tokens[i:i+n])
                    i += 2*n
                else:
                    out.append(tokens[i])
                    i += 1
        tokens = out
        return " ".join(tokens)

    def postprocess_one(self, text: str) -> str:
        t = str(text or "") # avoid None values --> no error

        t = self.remove_notes.sub("", t) # apply remove annotations -> ""
        
        t = self.gap_runs.sub("<big_gap>", t) # 2+ <gap> --> <big_gap>
        t = self.biggap_runs.sub("<big_gap>", t) # merge big gaps

        t = self.repeat_word.sub("", t) # apply remove repeat words
        t = self._dedupe_ngrams(t) # apply remove repeat phrases (ngram)

        # Space & Punctional Normalization (like in safe postprocessing)
        t = self.multi_punct.sub(r"\1", t)
        t = self.space_before_punct.sub(r"\1", t)
        t = self.multi_space.sub(" ", t).strip()

        return t
 
    def process_batch(self, texts:List[str]) -> List[str]:
        return (self.postpreprocess_one(text) for text in texts)
        

## Scoring translated

### Badness score (lower = better)

Include

- empty
- not enough/too many words
- many gaps
- repetitive words
- repetitive phrases

In [82]:
def badness_score(text: str) -> float:
    if text is None:
        text = ""

    t = str(text).strip()

    if not t:
        return 10.0 # empty

    words = t.split()
    n = len(words)
    score = 0.
    # Word count badness
    if n < 5:
        score += 3.0 # less than 5 words +3 bad
    if n < 3:
        score += 3.0 # less than 3 words +6 bad
    if n > 350:
        score += 3.0 # more than 350 words +3 bad
    if n > 500:
        score += 3.0 # more than 500 words +6 bad

    # Many gaps badness
    gaps = t.count("<gap>") + t.count("<big_gap>")
    if gaps > 6:
        score += (gaps - 6) * 0.35 # +0.35 per gap if gaps > 6
    
    # Many repetitive words badness
    for i in range (2, n):
        if words[i-2].lower() == words[i-1].lower() == word[i].lower:
            score += 0.75 # +0.75 per 3+ consecutive repeated words

    # Many repetitive phrases badness
    if n >= 20:
        bigrams = list(zip(words, words[1:]))
        uniq = len(set(bigrams)) # calculate the 1 - (unique bigrams / total bigrams) -> 1 - diversity
        if uniq > 0:
            repetitiveness = 1.0 - (uniq / max(1, len(bigrams)))
            if repetitiveness > 0.35:
                score += (repetitiveness - 0.35) * 6.0

    return score

### Quality score (higher = better)

Calculates how natrual of the output

Includes:

- Ideal length / Acceptable length
- Capitalized
- Well punctuation
- Include keywords
- Penalize gaps
- Apply badness


In [83]:
# These words are more common in Akkadian translit.
KEYWORDS = {
    "tablet", "king", "city", "god", "silver", "gold", "temple", "house", "palace",
    "year", "son", "daughter", "brother", "mother", "father", "gave", "took", "sent",
    "received", "grain", "sheep", "oil", "wine"
}

In [84]:
def quality_score(text: str) -> float:
    if text is None:
        text = ""
    t = str(text).strip()
    
    if not t:
        return 0.0 # so bad

    words = t.split()
    n = len(words)
    score = 0.0
    if 80 <= n <= 120:
        score += 2.0 # +3 if 80 <= len <= 120, check below
    if 50 <= n <= 200:
        score += 1.0 # +1 if 50 <= len <= 200

    # formatting
    if t[0].isupper():
        score += 0.5 # Uppercase the first word check
    if t.endswith((".", "?", "!")):
        score += 0.5 # Sentence ending punctation

    # keyword hints
    lw = {w.strip(".,;:!?\"'()[]").lower() for w in words}
    hit = len(lw.intersection(KEYWORDS))
    score += min(2.0, 0.25 * hit) # +0.25 per keyword, 2 max

    # penalize too many gaps
    gaps = t.count("<gap>") + t.count("<big_gap>")
    score -= min(2.0, 0.15 * gaps) # -0.15 per gap, 2 max

    # penalize heavy repetition by badness
    score -= 0.5 * max(0.0, badness_score(t) - 1.0)
    return score    

### Fuse texts (Final decision)

In [85]:
def fuse_texts(a: str, b: str, tie_badness_thresh=0.5, w_a=0.6, w_b=0.4, prefer="a") -> str:
    """
    Robust fuse:
      1) pick lower badness if clearly better
      2) if close, pick higher weighted quality
    """
    ba = badness_score(a)
    bb = badness_score(b)

    qa = quality_score(a) * w_a
    qb = quality_score(b) * w_b

    if ba + tie_badness_thresh < bb:
        return a
    if bb + tie_badness_thresh < ba:
        return b

    if qa > qb:
        return a
    if qa < qb:
        return b

    return a if prefer=="a" else b

## Dataset

loads dataset + preprocess

In [86]:
class AkkadianDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preprocessor: Preprocessor):
        # Extract Ids
        self.ids = df["id"].astype("str").tolist()

        # Extract translit. text
        raw = df["transliteration"].astype("object").fillna("").tolist()
        proc = preprocessor.preprocess_batch(raw) # process the raw input via Preprocessor class
        self.inputs = [f"translate Akkadian to English: {x}" for x in proc] # T5 instruction model

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], self.inputs[i]
        

## Bucket Sampler

Purpose: Merge samples has similar length into 1 batch --> faster inference

In [87]:
class BucketSampler(Sampler[List[int]]):
    def __init__(self, texts: List[str], batch_size: int, num_buckets=32, shuffle=False):
        self.batch_size = batch_size
        self.shuffle = shuffle

        lengths = np.array([max(1, len(t.split())) for t in texts], dtype=np.int32) # Measure text len.
        order = np.argsort(lengths) # Sort by len.
        self.indices = order.tolist()
        
        # Create buckets
        self.num_buckets = max(1, num_buckets)
        self.buckets = np.array_split(self.indices, self.num_buckets)

        # Split into batches
        self.batches = []
        rng = np.random.default_rng(42)
        for b in self.buckets:
            b = list(b)
            if shuffle:
                rng.shuffle(b)
            for i in range(0, len(b), batch_size): # split to batches in every buckets
                chunk = b[i:i+batch_size]
                if len(chunk) > 0:
                    self.batches.append(chunk) # push into self.batches

            if shuffle:
                rng.shuffle(self.batches)

    def __iter__(self):
        # Iteration
        for batch in self.batches:
            yield batch
            
    def __len__(self):
        # return number of batches
        return len(self.batches)


## Inference Engine

Includes

- load translation model
- prepares inputs
- runs generation
- clean the output
- return the prediction

In [88]:
class InferenceEngine:
    def __init__(self, cfg: CFG):
        # FIX: keep the runtime config instance (not the CFG class itself)
        self.cfg = cfg
        self.device = torch.device(cfg.device)
        self.preprocessor = Preprocessor()

        # Safe Postpreprocessor
        self.safe_pp = SafePostprocessor(use_unicode_fractions=False, strip_dash_old=False)
        self.safe_pp_stripdash = SafePostprocessor(use_unicode_fractions=False, strip_dash_old=True)
        # Aggressive Postprocessor
        self.agg_pp = AgressivePostprocessor()

        # Tokenizer + model
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.model_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_path).to(self.device)
        self.model.eval()

        # Optional BetterTransformer
        if cfg.use_better_transformer and cfg.device == "cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model = BetterTransformer.transform(self.model)
                logger.info("BetterTransformer enabled.")
            except Exception as e:
                logger.warning(f"BetterTransformer failed, continue without it: {e}")

    def _collate_fn(self, batch):
        ids, texts = zip(*batch)
        enc = self.tokenizer(
            list(texts),
            padding=True,
            truncation=True,  # FIX: typo (trunctation -> truncation)
            max_length=self.cfg.max_length,
            return_tensors="pt"
        )
        return list(ids), enc

    def _adaptive_beams(self, attention_mask: torch.Tensor) -> int:
        if not self.cfg.use_adaptive_beams:
            # FIX: return beam count, not boolean flag
            return self.cfg.num_beams

        lens = attention_mask.sum(dim=1).detach().cpu().numpy()
        med = float(np.median(lens)) if len(lens) else 0.0
        if med < 100:
            return max(4, self.cfg.num_beams // 2)
        return self.cfg.num_beams

    def _postprocess(self, texts: List[str]) -> List[str]:
        mode = self.cfg.postprocess_mode
        if mode == "safe_only":
            return self.safe_pp.postprocess_batch(texts)
        if mode == "aggressive_only":
            out = self.safe_pp_stripdash.postprocess_batch(texts)
            return self.agg_pp.postprocess_batch(out)

        safe = self.safe_pp.postprocess_batch(texts)
        thr = self.cfg.aggressive_trigger_badness
        refined = []
        for t in safe:
            refined.append(self.agg_pp.postprocess_one(t) if badness_score(t) >= thr else t)
        return refined

    @staticmethod
    def _final_fix_one(t: str, min_words=3) -> str:
        tt = (t or "").strip()
        if not tt:
            return "The tablet contains fragmentary text."

        words = tt.split()
        if len(words) < min_words:
            return "The tablet contains an incomplete inscription."

        # FIX: call islower() and use tt[0].upper() (not tt[0].upper)
        if tt and tt[0].isalpha() and tt[0].islower():
            tt = tt[0].upper() + tt[1:]

        if not tt.endswith((".", "!", "?")):
            tt = tt + "."

        tt = re.sub(r"\s+([,.;:!?])", r"", tt)
        tt = re.sub(r"\s+", " ", tt).strip()
        return tt

    def run_inference(self, test_df: pd.DataFrame, run_tag="run") -> pd.DataFrame:
        cfg = self.cfg
        ds = AkkadianDataset(test_df, self.preprocessor)

        if cfg.use_bucket_batching:
            sampler = BucketSampler(ds.inputs, batch_size=cfg.batch_size, num_buckets=32)
            dl = DataLoader(
                ds,
                batch_sampler=sampler,
                num_workers=cfg.num_workers,
                pin_memory=cfg.pin_memory,
                collate_fn=self._collate_fn
            )
        else:
            dl = DataLoader(
                ds,  # FIX: df -> ds
                batch_size=cfg.batch_size,
                shuffle=False,
                num_workers=cfg.num_workers,
                pin_memory=cfg.pin_memory,
                collate_fn=self._collate_fn
            )

        results: List[Tuple[str, str]] = []
        t0 = time.time()

        if cfg.use_mixed_precision and cfg.device == "cuda":
            autocast_ctx = torch.cuda.amp.autocast
        else:
            class _NullCtx:
                def __enter__(self): return None
                def __exit__(self, exc_type, exc, tb): return False
            autocast_ctx = lambda: _NullCtx()

        for step, (ids, enc) in enumerate(dl):
            input_ids = enc["input_ids"].to(self.device, non_blocking=True)
            attn = enc["attention_mask"].to(self.device, non_blocking=True)

            beams = self._adaptive_beams(attn)
            gen_kwargs = dict(
                num_beams=beams,
                max_new_tokens=cfg.max_new_tokens,
                length_penalty=cfg.length_penalty,
                early_stopping=cfg.early_stopping,
                no_repeat_ngram_size=cfg.no_repeat_ngram_size
            )
            if cfg.repetition_penalty is not None:
                gen_kwargs["repetition_penalty"] = cfg.repetition_penalty

            with torch.inference_mode():
                with autocast_ctx():
                    out_ids = self.model.generate(
                        input_ids=input_ids,
                        attention_mask=attn,
                        **gen_kwargs
                    )

            # FIX: very important -> remove special tokens during decode
            decoded = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
            processed = self._postprocess(decoded)
            processed = [self._final_fix_one(x, cfg.min_words_fallback) for x in processed]
            results.extend(list(zip(ids, processed)))

            if cfg.checkpoint_freq and (len(results) % cfg.checkpoint_freq == 0):
                ck = pd.DataFrame(results, columns=["id", "translation"])
                ck_path = os.path.join(cfg.output_dir, f"checkpoint_{run_tag}_{len(results)}.csv")
                ck.to_csv(ck_path, index=False)
                logger.info(f"Saved checkpoint: {ck_path}")

            if cfg.device == "cuda" and cfg.empty_cache_every and (step + 1) % cfg.empty_cache_every == 0:
                torch.cuda.empty_cache()

            if (step + 1) % 50 == 0:
                elapsed = time.time() - t0
                logger.info(f"[{run_tag}] step={step+1}/{len(dl)} | done={len(results)} | elapsed={elapsed:.1f}s")

        df_out = pd.DataFrame(results, columns=["id", "translation"])
        self._validate(df_out, run_tag=run_tag)
        return df_out

    def _validate(self, df: pd.DataFrame, run_tag: str):
        if df.empty:
            logger.warning(f"[{run_tag}] Empty output dataframe.")
            return

        lens = df["translation"].fillna("").map(lambda x: len(str(x).split()))
        empties = (df["translation"].fillna("").str.strip() == "").mean() * 100
        short = (lens < 5).sum()

        logger.info(
            f"[{run_tag}] rows={len(df)} | empty%={empties:.2f} | "
            f"len(mean/med/min/max)={lens.mean():.1f}/{lens.median():.1f}/{lens.min()}/{lens.max()} | "
            f"short(<5)={short}"
        )

        k = min(self.cfg.validate_samples, len(df))
        if k > 0:
            sample = df.sample(k, random_state=123)
            for _, r in sample.iterrows():
                logger.info(f"[{run_tag}] sample id={r['id']} | {str(r['translation'])[:160]}")


In [89]:
def make_cfg(base: CFG, preset: str, batch_size: Optional[int]=None, num_beams: Optional[int]=None, length_penalty: Optional[float]=None, repetition_penalty: Optional[float]=None, no_repeat_ngram_size: Optional[int]=None, strip_dash_old: Optional[bool]=None) -> CFG:
    cfg = CFG(**asdict(base)) # asdict for dataclass convert CFG to dictionary
    if preset == "baseline":
        cfg.length_penalty = 1.30
        cfg.repetition_penalty = None
    elif preset == "len115":
        cfg.length_penalty = 1.15
        cfg.repetition_penalty = None
    elif preset == "rep_pen":
        cfg.length_penalty = 1.30
        cfg.repetition_penalty = 1.08
    elif preset == "len115_rep":
        cfg.length_penalty = 1.15
        cfg.repetition_penalty = 1.08
    return cfg

def fuse_submissions(df_a: pd.DataFrame, df_b: pd.DataFrame, prefer="a", tie_badness_thresh=0.5, w_a=0.60, w_b=0.40) -> pd.DataFrame:
    a = df_a.set_index("id")["translation"] # Extract trans. col.
    b = df_b.set_index("id")["translation"]
    ids = a.index # Store Ids

    out = [] # Prepare output list

    for _id in ids: # Iterate over rows
        ta = a.loc[_id] # Retrieve predictions
        tb = b.loc[_id]
        fused = fuse_texts(
            ta,
            tb,
            prefer=prefer,
            tie_badness_thresh=tie_badness_thresh,
            w_a=w_a,
            w_b=w_b
        ) # Apply fuse text for #_id instance
        out.append((_id, fused)) # Store
    return pd.DataFrame(out, columns=["id", "translation"]) # Build final dataframe    
        

## Helpers to build configs (two-pass)

## Main: load test, run 2 passes, fuse, save

In [90]:
base_cfg = CFG() # Load the config
os.makedirs(base_cfg.output_dir, exist_ok=True) # Ensure is the output_dir exist

test_df = pd.read_csv(base_cfg.test_path) # Load test dataset
assert "id" in test_df.columns and "transliteration" in test_df.columns, "test.csv must have columns: id, transliteration" # Validate dataset format

# Save configuration snapshot
with open(os.path.join(base_cfg.output_dir, "run_config_base.json"), "w", encoding="utf-8") as f: # Save configuration snapshot
    json.dump(asdict(base_cfg), f, ensure_ascii=False, indent=2) # asdict convert CFG to dictionary

# Build two inference configs
# Pass A / Pass B presets (recommended defaults)
cfg_a = make_cfg(base_cfg, preset="len115")
cfg_b = make_cfg(base_cfg, preset="rep_pen")

logger.info(f"PassA: len_pen={cfg_a.length_penalty}, rep_pen={cfg_a.repetition_penalty}, beams={cfg_a.num_beams}, bs={cfg_a.batch_size}")
logger.info(f"PassB: len_pen={cfg_b.length_penalty}, rep_pen={cfg_b.repetition_penalty}, beams={cfg_b.num_beams}, bs={cfg_b.batch_size}")

engine_a = InferenceEngine(cfg_a) # Run inference past A
sub_a = engine_a.run_inference(test_df, run_tag="A_len115") # Run translation
path_a = os.path.join(cfg_a.output_dir, "submission_A.csv") # Save submission A
sub_a.to_csv(path_a, index=False) # Write CSV
logger.info(f"Saved: {path_a}")

# Free GPU memory a bit before second engine (optional)
del engine_a
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Run trans. B (same as A)
engine_b = InferenceEngine(cfg_b)
sub_b = engine_b.run_inference(test_df, run_tag="B_rep_pen")
path_b = os.path.join(cfg_b.output_dir, "submission_B.csv")
sub_b.to_csv(path_b, index=False)
logger.info(f"Saved: {path_b}")

# Fuse
fused = fuse_submissions(sub_a, sub_b, prefer="a", tie_badness_thresh=0.5, w_a=0.60, w_b=0.40)

# Final sanity (no empties)
fused["translation"] = fused["translation"].fillna("").map(lambda x: InferenceEngine._final_fix_one(str(x), min_words=base_cfg.min_words_fallback))

out_path = os.path.join(base_cfg.output_dir, "submission.csv")
fused.to_csv(out_path, index=False)
logger.info(f"Saved FINAL submission: {out_path}")

# Quick summary
lens = fused["translation"].map(lambda x: len(str(x).split()))
logger.info(f"FINAL | rows={len(fused)} | len(mean/med/min/max)={lens.mean():.1f}/{lens.median():.1f}/{lens.min()}/{lens.max()}")
print(fused.head())
print(fused.tail())


2026-03-13 15:16:22,275 | INFO | PassA: len_pen=1.15, rep_pen=None, beams=8, bs=8
2026-03-13 15:16:22,276 | INFO | PassB: len_pen=1.3, rep_pen=1.08, beams=8, bs=8
2026-03-13 15:16:23,136 | WARNING | BetterTransformer failed, continue without it: No module named 'optimum'
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Keyword arguments {'trunctation': True} not recognized.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Keyword arguments {'trunctation': True} not recognized.
Keyword arguments {'trunctation': True} not recognized.
Keyword arguments {'trunctation': True} not recognized.
/tmp/i

  id                            translation
0  3  The tablet contains fragmentary text.
1  0  The tablet contains fragmentary text.
2  1  The tablet contains fragmentary text.
3  2  The tablet contains fragmentary text.
  id                            translation
0  3  The tablet contains fragmentary text.
1  0  The tablet contains fragmentary text.
2  1  The tablet contains fragmentary text.
3  2  The tablet contains fragmentary text.
